# HomeReach Melbourne — 02: Clean the rental panel

**Week 2, milestone 2: turn the audited workbook into one row per LGA and quarter.**

Input: the unchanged September 2025 DFFH LGA workbook, worksheet `2br Flat`.
This notebook is **standalone**: it reconstructs the source mappings instead of
relying on variables left in the audit notebook's kernel. Keep `01_data_audit.ipynb`.

The intended analytical key is:

`rental_area_name + quarter + property_type`

For this snapshot, the source's three metropolitan groups contain 31 LGAs and
106 quarters, so the expected output is **3,286 rows** for `2 bedroom flats`.
`rental_area_name` is an **LGA label here**, not a suburb or a property address.

We will keep `Count` and `Median` in different columns, preserve source names and
Excel cell references, validate the complete output, and save CSV and Parquet.
No rent imputation, rent forecasting, geographic boundary join, or dashboard
replacement happens in this notebook. Original source files are never overwritten.

Run the cells from top to bottom with the **homereach** kernel. Do not run the
`.ipynb` file using Code Runner or `python notebook.ipynb`.


## 1. Imports, paths and source identity

`Path` finds the project; `hashlib` records the source's SHA-256 fingerprint.
`pandas` constructs the analytical table. All are already available in the
project environment. No new installation is needed.

The fingerprint below belongs to the exact workbook used in the completed audit.
A different file should trigger a review, not silently inherit its assumptions.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math
import sys

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 16)
pd.set_option("display.max_rows", 40)
pd.set_option("display.max_colwidth", 70)


def find_project_root(start: Path) -> Path:
    for folder in [start.resolve(), *start.resolve().parents]:
        if (folder / "pyproject.toml").is_file() and (folder / "data").is_dir():
            return folder
    raise FileNotFoundError("Open this notebook from the HomeReach project or its notebooks folder.")


PROJECT_ROOT = find_project_root(Path.cwd())
WORKBOOK_NAME = (
    "quarterly-median-rents-local-government-area-"
    "september-quarter-2025-excel.xlsx"
)
WORKBOOK_PATH = PROJECT_ROOT / "data" / "raw" / "rent" / WORKBOOK_NAME
SHEET_NAME = "2br Flat"
PROPERTY_TYPE = "2 bedroom flats"
SOURCE_REPORTING_QUARTER = "2025Q3"
EXPECTED_SHA256 = "b8e96347494f09fea2a2ace55997fd34c9398f8509cb3450d65444aebfaaa837"
SOURCE_PAGE = "https://www.dffh.vic.gov.au/publications/rental-report"

if not WORKBOOK_PATH.is_file():
    raise FileNotFoundError(f"Put the unchanged workbook here: {WORKBOOK_PATH}")

source_sha256 = hashlib.sha256(WORKBOOK_PATH.read_bytes()).hexdigest()
if source_sha256 != EXPECTED_SHA256:
    raise ValueError("This workbook differs from the audited snapshot. Re-audit it before cleaning.")

print("Python:", sys.executable)
print("pandas:", pd.__version__)
print("Project root:", PROJECT_ROOT)
print("Workbook found:", WORKBOOK_PATH.is_file())
print("Matches audited source fingerprint:", source_sha256 == EXPECTED_SHA256)


Python: c:\Users\PhamDang\anaconda3\envs\homereach\python.exe
pandas: 3.0.3
Project root: D:\My project\homereach_melbourne
Workbook found: True
Matches audited source fingerprint: True


## 2. Load the source without altering it

Keep the same read settings used in the audit: `header=None` preserves all rows,
`dtype=object` avoids guessing a single type for each mixed-layout column, and
`keep_default_na=False` lets us inspect source blanks and hyphens explicitly.
The file is read only; no Excel-writing operation is used.


In [2]:
raw = pd.read_excel(
    WORKBOOK_PATH,
    sheet_name=SHEET_NAME,
    header=None,
    dtype=object,
    keep_default_na=False,
    engine="openpyxl",
)

if raw.shape != (95, 214):
    raise ValueError(f"Expected audited shape (95, 214); received {raw.shape}.")
if str(raw.iat[1, 0]).strip() != PROPERTY_TYPE:
    raise ValueError("The property-category label in A2 does not match the audit.")

print("Worksheet:", SHEET_NAME)
print("Source property label:", raw.iat[1, 0])
print("Raw worksheet shape:", raw.shape)


Worksheet: 2br Flat
Source property label: 2 bedroom flats
Raw worksheet shape: (95, 214)


## 3. Select the source's 31 metropolitan LGAs

Only the regional metadata is forward-filled because the workbook prints each
regional heading once. Rent and count values are **never** forward-filled.

We exclude `Group Total`, `Metro`, `Non-Metro` and `Victoria`, then select
`North and West Metro`, `Eastern Metro` and `Southern Metro`.
This is the publisher's grouping for this stage, not an assertion that every
other geographic classification uses identical boundaries.

Original LGA spelling, including `Mornington Penin'a`, is retained. We will
review and map names when introducing geographic boundaries, not guess now.


In [3]:
AGGREGATE_LABELS = {"Group Total", "Metro", "Non-Metro", "Victoria"}
METRO_GROUP_COUNTS = {
    "North and West Metro": 14,
    "Eastern Metro": 7,
    "Southern Metro": 10,
}

metadata = raw.iloc[3:, :2].copy()
metadata.columns = ["source_region", "rental_area_name"]
metadata["source_excel_row"] = metadata.index + 1

for column in ["source_region", "rental_area_name"]:
    metadata[column] = metadata[column].astype("string").str.strip().replace("", pd.NA)
metadata["source_region"] = metadata["source_region"].ffill()

all_lgas = metadata.loc[
    metadata["rental_area_name"].notna()
    & ~metadata["rental_area_name"].isin(AGGREGATE_LABELS)
].copy()
metro_lgas = all_lgas.loc[
    all_lgas["source_region"].isin(METRO_GROUP_COUNTS)
].copy()

if len(all_lgas) != 79 or len(metro_lgas) != 31:
    raise ValueError("The individual-LGA inventory differs from the audited snapshot.")
if metro_lgas["rental_area_name"].duplicated().any():
    raise ValueError("Duplicate metropolitan LGA names require review.")
if metro_lgas.groupby("source_region").size().to_dict() != METRO_GROUP_COUNTS:
    raise ValueError("Metropolitan regional group sizes differ from the audit.")

print("Selected metropolitan LGAs:", len(metro_lgas))
display(metro_lgas.reset_index(drop=True))


Selected metropolitan LGAs: 31


,source_region,rental_area_name,source_excel_row
0,North and West Metro,Banyule,57
1,North and West Metro,Brimbank,58
2,North and West Metro,Darebin,59
3,North and West Metro,Hobsons Bay,60
4,North and West Metro,Hume,61
5,North and West Metro,Maribyrnong,62
6,North and West Metro,Melbourne,63
7,North and West Metro,Melton,64
8,North and West Metro,Merri-bek,65
9,North and West Metro,Moonee Valley,66


## 4. Rebuild and validate the quarter/measure map

Each quarter has an adjacent `Count` column and `Median` column. We check both
header rows instead of merely assuming every second column is rent.

Excel column letters and row numbers will become source references in the
cleaned table. Pandas positions begin at zero; Excel row numbers begin at one.


In [4]:
def excel_column_name(position: int) -> str:
    if position < 0:
        raise ValueError("Column position cannot be negative.")
    number, letters = position + 1, ""
    while number:
        number, remainder = divmod(number - 1, 26)
        letters = chr(65 + remainder) + letters
    return letters


quarter_rows = []
if (raw.shape[1] - 2) % 2:
    raise ValueError("There is an unpaired measurement column.")

for count_pos in range(2, raw.shape[1], 2):
    median_pos = count_pos + 1
    count_label = " ".join(str(raw.iat[1, count_pos]).split())
    median_label = " ".join(str(raw.iat[1, median_pos]).split())
    measures = (
        str(raw.iat[2, count_pos]).strip(),
        str(raw.iat[2, median_pos]).strip(),
    )
    if count_label != median_label or measures != ("Count", "Median"):
        raise ValueError(f"Unexpected header pair at {excel_column_name(count_pos)}.")

    date = pd.to_datetime(count_label, format="%b %Y", errors="raise")
    if date.month not in {3, 6, 9, 12}:
        raise ValueError(f"Not a quarter-ending month: {count_label}")
    quarter_rows.append({
        "quarter": str(date.to_period("Q")),
        "source_quarter_label": count_label,
        "count_column_position": count_pos,
        "median_column_position": median_pos,
        "count_excel_column": excel_column_name(count_pos),
        "median_excel_column": excel_column_name(median_pos),
    })

quarter_map = pd.DataFrame(quarter_rows)
expected_periods = pd.period_range("1999Q2", "2025Q3", freq="Q")
actual_periods = pd.PeriodIndex(quarter_map["quarter"], freq="Q")
if not actual_periods.equals(expected_periods):
    raise ValueError("Quarter headers have a gap, duplicate, ordering issue or changed coverage.")

print("Quarters:", len(quarter_map))
print("Quarter range:", quarter_map.iloc[0]["quarter"], "to", quarter_map.iloc[-1]["quarter"])
display(pd.concat([quarter_map.head(2), quarter_map.tail(2)], ignore_index=True))


Quarters: 106
Quarter range: 1999Q2 to 2025Q3


,quarter,source_quarter_label,count_column_position,median_column_position,count_excel_column,median_excel_column
0,1999Q2,Jun 1999,2,3,C,D
1,1999Q3,Sep 1999,4,5,E,F
2,2025Q2,Jun 2025,210,211,HC,HD
3,2025Q3,Sep 2025,212,213,HE,HF


## 5. Build one quarter first, then stack all quarters

For a single quarter, copy the 31 LGA metadata rows and attach that quarter's
count and median values. This makes a 31-row table. `pd.concat(...,
ignore_index=True)` then stacks the 106 quarter tables into a long table.

Using `.to_numpy()` below preserves the order selected by `.loc` without pandas
trying to realign Series labels during assignment. We select source rows in
exactly the same order as the copied metadata.

There is no need to merge counts and medians by a guessed column name: each
validated quarter pair supplies both measures to the same row.


In [5]:
def build_quarter_table(mapping) -> pd.DataFrame:
    part = metro_lgas.copy()
    part["quarter"] = mapping.quarter
    part["property_type"] = PROPERTY_TYPE
    part["published_count"] = raw.loc[metro_lgas.index].iloc[
        :, mapping.count_column_position
    ].to_numpy()
    part["median_weekly_rent"] = raw.loc[metro_lgas.index].iloc[
        :, mapping.median_column_position
    ].to_numpy()
    part["source_count_cell"] = (
        mapping.count_excel_column + part["source_excel_row"].astype(str)
    )
    part["source_median_cell"] = (
        mapping.median_excel_column + part["source_excel_row"].astype(str)
    )
    return part


quarter_parts = [
    build_quarter_table(mapping)
    for mapping in quarter_map.itertuples(index=False)
]
print("Rows in one quarter:", len(quarter_parts[-1]))
display(quarter_parts[-1].head(5))

panel = pd.concat(quarter_parts, ignore_index=True)
print("Rows after stacking all quarters:", len(panel))


Rows in one quarter: 31


,source_region,rental_area_name,source_excel_row,quarter,property_type,published_count,median_weekly_rent,source_count_cell,source_median_cell
56,North and West Metro,Banyule,57,2025Q3,2 bedroom flats,282,540,HE57,HF57
57,North and West Metro,Brimbank,58,2025Q3,2 bedroom flats,224,400,HE58,HF58
58,North and West Metro,Darebin,59,2025Q3,2 bedroom flats,473,530,HE59,HF59
59,North and West Metro,Hobsons Bay,60,2025Q3,2 bedroom flats,149,500,HE60,HF60
60,North and West Metro,Hume,61,2025Q3,2 bedroom flats,115,450,HE61,HF61


Rows after stacking all quarters: 3286


## 6. Convert measures deliberately and test the conversion

`published_count` retains the source's `Count` measure as a nullable integer.
`median_weekly_rent` retains `Median` in AUD per week as a floating-point number.
We do not rename the count to available properties, stock or vacancies.

The audited source markers `-` and blank become missing values. Unexpected text,
negative values, non-finite numbers or fractional counts cause an error rather
than being silently removed. We test these behaviours even though this
metropolitan subset currently has no missing observations.


In [6]:
def parse_published_measure(values: pd.Series, *, integer: bool) -> pd.Series:
    text = values.astype("string").str.strip().replace({"": pd.NA, "-": pd.NA})
    numeric = pd.to_numeric(text, errors="coerce")
    unexpected = text.notna() & numeric.isna()
    if unexpected.any():
        examples = text.loc[unexpected].unique().tolist()[:5]
        raise ValueError(f"Unexpected source text requires review: {examples}")

    present = numeric.dropna()
    if not all(math.isfinite(float(value)) for value in present):
        raise ValueError("Non-finite numeric values require review.")
    if (present < 0).any():
        raise ValueError("Negative count or rent values require review.")
    if integer:
        if present.mod(1).ne(0).any():
            raise ValueError("Published counts must be whole numbers.")
        return numeric.astype("Int64")
    return numeric.astype("Float64").astype("float64")


# Small self-checks: missing is not zero, counts are integers, bad tokens fail.
probe = parse_published_measure(pd.Series([12, "-", "", "0"]), integer=True)
assert probe.iloc[0] == 12
assert pd.isna(probe.iloc[1]) and pd.isna(probe.iloc[2])
assert probe.iloc[3] == 0  # a real zero stays zero; a missing value does not become zero
assert str(probe.dtype) == "Int64"
assert parse_published_measure(pd.Series([600.5]), integer=False).iloc[0] == 600.5

for invalid_values, integer in [(["unknown"], False), ([-5], False), ([2.5], True), (["inf"], False)]:
    try:
        parse_published_measure(pd.Series(invalid_values), integer=integer)
    except ValueError:
        pass
    else:
        raise AssertionError(f"The parser should have rejected {invalid_values}.")

panel["published_count"] = parse_published_measure(panel["published_count"], integer=True)
panel["median_weekly_rent"] = parse_published_measure(panel["median_weekly_rent"], integer=False)

print("Numeric-conversion self-checks passed.")
print("Missing counts:", int(panel["published_count"].isna().sum()))
print("Missing rents:", int(panel["median_weekly_rent"].isna().sum()))


Numeric-conversion self-checks passed.
Missing counts: 0
Missing rents: 0


## 7. Add dates, provenance and a stable column order

Store `quarter` as a portable `YYYYQn` string and `quarter_end` as a real date
for plotting and sorting. `2025Q3`, for example, ends on 2025-09-30.

`source_reporting_quarter` describes the **workbook snapshot**, not every row's
observation quarter. Its historical entries were read from the September 2025
publication; this is not evidence that an identical historical vintage was
available at each date in the past. Later model evaluation must consider that.

`rental_area_name` matches the starter project's naming convention. `area_type`
explicitly identifies these records as LGAs. No official numeric area code is
invented, and the original published LGA name is preserved.


In [7]:
panel["quarter_end"] = (
    pd.PeriodIndex(panel["quarter"], freq="Q")
    .to_timestamp(how="end")
    .normalize()
)
panel["quarter_end"] = panel["quarter_end"].astype("datetime64[ns]")
panel["area_type"] = "LGA"
panel["source_file"] = WORKBOOK_PATH.name
panel["source_sheet"] = SHEET_NAME
panel["source_reporting_quarter"] = SOURCE_REPORTING_QUARTER
panel["source_excel_row"] = panel["source_excel_row"].astype("int64")

TEXT_COLUMNS = [
    "rental_area_name", "source_region", "area_type", "property_type", "quarter",
    "source_file", "source_sheet", "source_reporting_quarter",
    "source_count_cell", "source_median_cell",
]
for column in TEXT_COLUMNS:
    panel[column] = panel[column].astype("string")

COLUMN_ORDER = [
    "rental_area_name", "source_region", "area_type", "property_type",
    "quarter", "quarter_end", "published_count", "median_weekly_rent",
    "source_file", "source_sheet", "source_reporting_quarter", "source_excel_row",
    "source_count_cell", "source_median_cell",
]
panel = panel[COLUMN_ORDER].sort_values(
    ["rental_area_name", "property_type", "quarter_end"], kind="stable"
).reset_index(drop=True)

print("Cleaned table shape:", panel.shape)
display(panel[["rental_area_name", "quarter", "quarter_end", "published_count", "median_weekly_rent"]].head(8))
display(panel.dtypes.rename("dtype").to_frame())


Cleaned table shape: (3286, 14)


,rental_area_name,quarter,quarter_end,published_count,median_weekly_rent
0,Banyule,1999Q2,1999-06-30,214,165.0
1,Banyule,1999Q3,1999-09-30,202,165.0
2,Banyule,1999Q4,1999-12-31,232,170.0
3,Banyule,2000Q1,2000-03-31,271,170.0
4,Banyule,2000Q2,2000-06-30,208,171.0
5,Banyule,2000Q3,2000-09-30,204,175.0
6,Banyule,2000Q4,2000-12-31,225,175.0
7,Banyule,2001Q1,2001-03-31,261,180.0


,dtype
rental_area_name,string
source_region,string
area_type,string
property_type,string
quarter,string
quarter_end,datetime64[ns]
published_count,Int64
median_weekly_rent,float64
source_file,string
source_sheet,string


## 8. Validate every record before writing files

A correct total row count alone is insufficient. We check the unique key,
expected LGA-quarter combinations, dates, types, missingness and geography.

We also compare **all 6,572 cleaned numeric values** (3,286 counts and 3,286 rents)
back to their original worksheet cells. A source-cell reference is not merely a
label: this test verifies that it actually points to the preserved value.

These assertions deliberately describe this audited snapshot. Revisit them
when a new source is introduced; do not simply delete a failing check.


In [8]:
KEY_COLUMNS = ["rental_area_name", "quarter", "property_type"]
expected_quarter_labels = expected_periods.astype(str).tolist()
expected_lga_names = sorted(metro_lgas["rental_area_name"].tolist())


def excel_column_position(cell_address: str) -> int:
    letters = "".join(character for character in cell_address if character.isalpha())
    position = 0
    for letter in letters:
        position = position * 26 + ord(letter.upper()) - ord("A") + 1
    return position - 1


def validate_panel(frame: pd.DataFrame) -> dict:
    assert frame.columns.tolist() == COLUMN_ORDER, "Unexpected output schema."
    assert len(frame) == 3286, "Expected 31 LGAs x 106 quarters."
    assert frame[KEY_COLUMNS].notna().all().all(), "A key contains missing values."
    assert not frame.duplicated(KEY_COLUMNS).any(), "Duplicate LGA-quarter-category key."
    assert sorted(frame["rental_area_name"].unique().tolist()) == expected_lga_names
    assert frame["property_type"].eq(PROPERTY_TYPE).all()
    assert frame["area_type"].eq("LGA").all()
    assert not frame["rental_area_name"].isin(AGGREGATE_LABELS).any()
    assert frame["source_region"].isin(METRO_GROUP_COUNTS).all()
    assert frame["source_file"].eq(WORKBOOK_NAME).all()
    assert frame["source_sheet"].eq(SHEET_NAME).all()
    assert frame["source_reporting_quarter"].eq(SOURCE_REPORTING_QUARTER).all()
    assert pd.api.types.is_datetime64_any_dtype(frame["quarter_end"])
    assert pd.api.types.is_integer_dtype(frame["published_count"])
    assert pd.api.types.is_numeric_dtype(frame["median_weekly_rent"])
    assert frame[["published_count", "median_weekly_rent"]].notna().all().all()
    assert frame["published_count"].gt(0).all()
    assert frame["median_weekly_rent"].gt(0).all()

    expected_dates = pd.PeriodIndex(frame["quarter"], freq="Q").to_timestamp(how="end").normalize()
    assert pd.DatetimeIndex(frame["quarter_end"]).equals(expected_dates)
    for _, group in frame.groupby(["rental_area_name", "property_type"], sort=False):
        assert group.sort_values("quarter_end")["quarter"].tolist() == expected_quarter_labels

    compared_values = 0
    for row in frame.itertuples(index=False):
        raw_row = int(row.source_excel_row) - 1
        assert str(raw.iat[raw_row, 1]).strip() == row.rental_area_name
        assert metadata.loc[raw_row, "source_region"] == row.source_region
        for cell_address, value, measure in [
            (row.source_count_cell, row.published_count, "Count"),
            (row.source_median_cell, row.median_weekly_rent, "Median"),
        ]:
            cell_row = int("".join(character for character in cell_address if character.isdigit())) - 1
            cell_column = excel_column_position(cell_address)
            assert cell_row == raw_row
            assert str(raw.iat[2, cell_column]).strip() == measure
            label = " ".join(str(raw.iat[1, cell_column]).split())
            source_period = str(pd.to_datetime(label, format="%b %Y").to_period("Q"))
            assert source_period == row.quarter
            assert float(raw.iat[cell_row, cell_column]) == float(value)
            compared_values += 1

    assert compared_values == 6572
    assert hashlib.sha256(WORKBOOK_PATH.read_bytes()).hexdigest() == source_sha256
    return {
        "rows": int(len(frame)),
        "columns": int(frame.shape[1]),
        "lga_count": int(frame["rental_area_name"].nunique()),
        "quarter_count": int(frame["quarter"].nunique()),
        "first_quarter": str(frame["quarter"].min()),
        "last_quarter": str(frame["quarter"].max()),
        "duplicate_keys": int(frame.duplicated(KEY_COLUMNS).sum()),
        "missing_count_values": int(frame["published_count"].isna().sum()),
        "missing_rent_values": int(frame["median_weekly_rent"].isna().sum()),
        "source_values_verified": compared_values,
        "source_unchanged": True,
    }


quality = validate_panel(panel)
print("All panel validation checks passed.")
print(json.dumps(quality, indent=2))

# Prove that duplicate keys are rejected rather than silently dropped.
try:
    validate_panel(pd.concat([panel.iloc[[1]], panel.iloc[1:]], ignore_index=True))
except AssertionError:
    print("Duplicate-key rejection self-check passed.")
else:
    raise AssertionError("The validator failed to reject a duplicate key.")


All panel validation checks passed.
{
  "rows": 3286,
  "columns": 14,
  "lga_count": 31,
  "quarter_count": 106,
  "first_quarter": "1999Q2",
  "last_quarter": "2025Q3",
  "duplicate_keys": 0,
  "missing_count_values": 0,
  "missing_rent_values": 0,
  "source_values_verified": 6572,
  "source_unchanged": true
}
Duplicate-key rejection self-check passed.


## 9. Read three understandable examples

These three rows are a source-reading check, **not recommendations or current
listings**. They describe `2 bedroom flats` in September 2025. `Melbourne` is
one LGA in this table, not all metropolitan Melbourne.

Keep the count and weekly rent separate. Do not calculate a metropolitan rent
by averaging LGA medians: that is not the median of the underlying rentals.


In [9]:
latest_examples = panel.loc[
    panel["quarter"].eq("2025Q3")
    & panel["rental_area_name"].isin(["Melbourne", "Monash", "Whitehorse"]),
    ["rental_area_name", "property_type", "quarter", "published_count", "median_weekly_rent", "source_count_cell", "source_median_cell"],
].reset_index(drop=True)

expected_examples = {"Melbourne": (3209, 730), "Monash": (421, 600), "Whitehorse": (464, 600)}
for row in latest_examples.itertuples(index=False):
    assert (int(row.published_count), float(row.median_weekly_rent)) == expected_examples[row.rental_area_name]
assert len(latest_examples) == 3

display(latest_examples)


,rental_area_name,property_type,quarter,published_count,median_weekly_rent,source_count_cell,source_median_cell
0,Melbourne,2 bedroom flats,2025Q3,3209,730.0,HE63,HF63
1,Monash,2 bedroom flats,2025Q3,421,600.0,HE76,HF76
2,Whitehorse,2 bedroom flats,2025Q3,464,600.0,HE77,HF77


## 10. Save CSV and Parquet, then read them back

Parquet is the typed analytical file that later Python and SQL steps will load.
CSV is an inspection/export copy; CSV itself does not preserve a schema, so
we supply text/integer/date types explicitly when reading it back for the test.
Both files contain the same 3,286 rows and 14 columns.

Only these **derived** files are replaced on a successful rerun. The raw Excel
workbook remains untouched. `data/processed/` is intended for reproducible local
outputs; these files are not automatically a new input to the demo application.


In [10]:
try:
    import pyarrow
except ImportError as error:
    raise ImportError(
        "Parquet requires pyarrow, which is listed in environment.yml. "
        "First check that this notebook uses the homereach kernel."
    ) from error

OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PARQUET_PATH = OUTPUT_DIR / "rent_quarterly.parquet"
CSV_PATH = OUTPUT_DIR / "rent_quarterly.csv"

# Validate immediately before writing, even when this cell is rerun alone.
quality = validate_panel(panel)
panel.to_parquet(PARQUET_PATH, index=False, engine="pyarrow")
panel.to_csv(CSV_PATH, index=False, date_format="%Y-%m-%d", encoding="utf-8")

parquet_check = pd.read_parquet(PARQUET_PATH, engine="pyarrow")
csv_types = {column: "string" for column in TEXT_COLUMNS}
csv_types.update({"published_count": "Int64", "median_weekly_rent": "float64", "source_excel_row": "int64"})
csv_check = pd.read_csv(
    CSV_PATH, dtype=csv_types, parse_dates=["quarter_end"], keep_default_na=False,
)

csv_check["quarter_end"] = csv_check["quarter_end"].astype("datetime64[ns]")

for label, reloaded in [("Parquet", parquet_check), ("CSV", csv_check)]:
    validate_panel(reloaded)
    # Storage backends may differ for strings/datetimes across pandas versions;
    # the semantic types are checked above, and all values must remain exact.
    pd.testing.assert_frame_equal(panel, reloaded, check_dtype=False, check_exact=True)
    print(f"{label} read-back matches: True")

assert hashlib.sha256(WORKBOOK_PATH.read_bytes()).hexdigest() == source_sha256
print("Saved:", PARQUET_PATH.relative_to(PROJECT_ROOT))
print("Saved:", CSV_PATH.relative_to(PROJECT_ROOT))


Parquet read-back matches: True
CSV read-back matches: True
Saved: data\processed\rent_quarterly.parquet
Saved: data\processed\rent_quarterly.csv


## 11. Record the cleaning method and final checkpoint

This cell writes a machine-readable quality report and a short Markdown report
with a data dictionary. It does not overwrite the audit, project charter, Week 1
progress file or existing data-dictionary template.

The report documents what was checked, not a claim that the data is free of every
bias or ready for real-world recommendations. We have **not** tested historical
geographic consistency, inflation adjustment, model performance or live availability.


In [11]:
assert PARQUET_PATH.is_file() and CSV_PATH.is_file()
quality = validate_panel(panel)
created_at_utc = datetime.now(timezone.utc).isoformat()
manifest = {
    "created_at_utc": created_at_utc,
    "pandas_version": pd.__version__,
    "source": {
        "filename": WORKBOOK_NAME,
        "publisher_page": SOURCE_PAGE,
        "sha256": source_sha256,
        "worksheet": SHEET_NAME,
        "property_label": PROPERTY_TYPE,
        "reporting_quarter": SOURCE_REPORTING_QUARTER,
    },
    "geography": {"definition": "Source's three metropolitan groups; individual LGA rows only", "groups": METRO_GROUP_COUNTS},
    "analytical_key": KEY_COLUMNS,
    "quality": quality,
    "parquet_readback_matches": True,
    "csv_readback_matches": True,
    "columns": [{"name": column, "dtype": str(panel[column].dtype)} for column in panel.columns],
    "outputs": [str(path.relative_to(PROJECT_ROOT)).replace("\\", "/") for path in [PARQUET_PATH, CSV_PATH]],
}
manifest_path = OUTPUT_DIR / "rent_quality_report.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

report_path = PROJECT_ROOT / "docs" / "rental_data_cleaning.md"
report_path.parent.mkdir(parents=True, exist_ok=True)
report_lines = [
    "# Rental data cleaning — HomeReach Melbourne", "",
    "## Source and output", "",
    f"- Source file: `{WORKBOOK_NAME}`",
    f"- Source SHA-256: `{source_sha256}`",
    f"- Publisher: {SOURCE_PAGE}",
    f"- Source worksheet: `{SHEET_NAME}`; published property label: `{PROPERTY_TYPE}`.",
    f"- Source reporting snapshot: `{SOURCE_REPORTING_QUARTER}`.",
    f"- Generated UTC: {created_at_utc}",
    "- Output: `data/processed/rent_quarterly.parquet` and `rent_quarterly.csv`.",
    "- Quality manifest: `data/processed/rent_quality_report.json`.", "",
    "## Analytical grain and coverage", "",
    "One row = one source-labelled LGA x one quarter x one property category.",
    "Key: `rental_area_name`, `quarter`, `property_type`.",
    "The output has 3,286 rows and 14 columns: 31 LGAs x 106 quarters, 1999Q2–2025Q3.",
    "The metropolitan subset uses the publisher's North and West Metro (14),",
    "Eastern Metro (7), and Southern Metro (10) groups. Totals are excluded.", "",
    "## Transformation", "",
    "1. Read the source without editing it; verify its audited fingerprint and headers.",
    "2. Forward-fill regional metadata only and retain original LGA names.",
    "3. Select individual LGAs in the three source metropolitan groups.",
    "4. Build one 31-row table for each validated Count/Median quarter pair.",
    "5. Stack all quarter tables, convert measures explicitly and add real quarter-end dates.",
    "6. Retain source row, count-cell and rent-cell references; sort by LGA, category and date.",
    "7. Validate keys, coverage, types, missingness, all source values and file read-back.", "",
    "## Data dictionary", "",
    "| Column | Meaning |",
    "|---|---|",
    "| rental_area_name | Original LGA label; not a suburb or address. |",
    "| source_region | Source metropolitan group label. |",
    "| area_type | LGA. |",
    "| property_type | Original label: 2 bedroom flats. |",
    "| quarter | Observation quarter, stored as YYYYQn. |",
    "| quarter_end | Calendar quarter-end date. |",
    "| published_count | Source Count measure, stored as a nullable integer; not available stock. |",
    "| median_weekly_rent | Source Median, in AUD per week. |",
    "| source_file | Original workbook filename, without a local personal path. |",
    "| source_sheet | 2br Flat. |",
    "| source_reporting_quarter | Workbook publication snapshot: 2025Q3. |",
    "| source_excel_row | One-based original worksheet row. |",
    "| source_count_cell | Original Count cell address. |",
    "| source_median_cell | Original Median cell address. |", "",
    "## Validation outcome", "",
    f"- Duplicate keys: {quality['duplicate_keys']}.",
    f"- Missing count values: {quality['missing_count_values']}.",
    f"- Missing rent values: {quality['missing_rent_values']}.",
    f"- Numeric values checked against original cells: {quality['source_values_verified']}.",
    "- Every LGA has 106 consecutive quarters in the selected snapshot.",
    "- CSV and Parquet read-back match the in-memory table.",
    "- Original workbook fingerprint unchanged.", "",
    "## Limitations and deferred work", "",
    "These are area-level historical published statistics from a September 2025 snapshot,",
    "not live listings, vacancies or current prices. Count is retained under its original",
    "measure name; consult the publisher's definitions before further interpretation.",
    "Do not obtain the metropolitan median by averaging LGA medians.",
    "Missing source markers would stay missing; no rent or count values were imputed.",
    "Names such as Mornington Penin'a are preserved for a later reviewed geography crosswalk.",
    "Source grouping and geographic vintages still need review when joining boundary data.",
    "Historical values in this snapshot are not proof of identical real-time historical vintages.",
    "No transport join, rent-growth feature, forecast, inflation adjustment or dashboard replacement",
    "has been performed. The demo app still uses synthetic data until deliberately connected.", "",
]
report_path.write_text("\n".join(report_lines), encoding="utf-8")
assert hashlib.sha256(WORKBOOK_PATH.read_bytes()).hexdigest() == source_sha256

print("CLEANING CHECKPOINT PASSED")
print(f"Rows: {quality['rows']}")
print(f"Columns: {quality['columns']}")
print(f"Metro LGAs: {quality['lga_count']}")
print(f"Quarters: {quality['quarter_count']} ({quality['first_quarter']} to {quality['last_quarter']})")
print(f"Duplicate keys: {quality['duplicate_keys']}")
print(f"Missing count values: {quality['missing_count_values']}")
print(f"Missing rent values: {quality['missing_rent_values']}")
print(f"Source numeric values verified: {quality['source_values_verified']}")
print("CSV and Parquet read-back: matched")
print("Original workbook unchanged: True")
print("Saved:", manifest_path.relative_to(PROJECT_ROOT))
print("Saved:", report_path.relative_to(PROJECT_ROOT))


CLEANING CHECKPOINT PASSED
Rows: 3286
Columns: 14
Metro LGAs: 31
Quarters: 106 (1999Q2 to 2025Q3)
Duplicate keys: 0
Missing count values: 0
Missing rent values: 0
Source numeric values verified: 6572
CSV and Parquet read-back: matched
Original workbook unchanged: True
Saved: data\processed\rent_quality_report.json
Saved: docs\rental_data_cleaning.md


## Checkpoint and next step

Save this notebook with **Ctrl+S**. Keep the original audit notebook too.

Expected final outcome: **3,286 rows, 31 LGAs, 106 quarters, zero duplicate keys,
zero missing counts/rents, and 6,572 numeric values matched to source cells**.
This is a validated source-specific rental panel, not a completed model.

Next we will move the proven parsing/validation logic into a reusable Python
module, add automated tests that do not require uploading the raw workbook,
and update the data registry. After that we can build the first real-data
rental analyses and geographic joins. No need to change the Streamlit demo yet.

### What you should be able to explain

- Why 31 LGAs and 106 quarters produce 3,286 rows rather than 6,572 rows.
- Why each row has two different numeric measures.
- How a row's Excel cell references verify its values.
- Why missing values must not become zero and totals are excluded.
- Why the observation quarter and the source reporting snapshot are different.

### Technical references

- pandas Excel loading: https://pandas.pydata.org/docs/reference/api/pandas.read_excel.html
- pandas concatenation: https://pandas.pydata.org/docs/reference/api/pandas.concat.html
- pandas quarterly timestamps: https://pandas.pydata.org/docs/reference/api/pandas.PeriodIndex.to_timestamp.html
- pandas Parquet writing: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_parquet.html
- pandas frame comparisons: https://pandas.pydata.org/docs/reference/api/pandas.testing.assert_frame_equal.html
